In [ ]:
#3.1基本用法
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

# 初始化处理器
dit_handler = AceStepHandler()
llm_handler = LLMHandler()

# 初始化服务
dit_handler.initialize_service(
    project_root="/srv/home/guxin/ACE-Step-1.5-main",
    config_path="acestep-v15-turbo",
    device="cuda"
)

llm_handler.initialize(
    checkpoint_dir="/srv/home/guxin/ACE-Step-1.5-main/checkpoints",
    lm_model_path="acestep-5Hz-lm-4B",
    backend="vllm",
    device="cuda"
)


2026-02-07 15:23:06.196 | INFO     | acestep.handler:initialize_service:399 - [initialize_service] Attempting to load model with attention implementation: sdpa
2026-02-07 15:23:10.384 | INFO     | acestep.llm_inference:initialize:364 - loading 5Hz LM tokenizer... it may take 80~90s
2026-02-07 15:23:33.205 | INFO     | acestep.llm_inference:initialize:368 - 5Hz LM tokenizer loaded successfully in 22.82 seconds
2026-02-07 15:23:33.209 | INFO     | acestep.llm_inference:initialize:373 - Initializing constrained decoding processor...
2026-02-07 15:23:33.210 | INFO     | acestep.llm_inference:initialize:379 - Setting constrained decoding max_duration to 600s based on GPU config (tier: unlimited)
2026-02-07 15:23:34.628 | WARNING  | acestep.constrained_logits_processor:_precompute_audio_code_tokens:556 - Found 1535 audio code tokens with values outside valid range [0, 63999]
2026-02-07 15:23:38.520 | INFO     | acestep.llm_inference:initialize:387 - Constrained processor initialized in 5.31 

模型选择：


In [ ]:
# 3.1基本用法--生成音乐

# 配置生成参数
params = GenerationParams(
    caption="欢快的电子舞曲，重低音",
    bpm=128,
    duration=30,
)

# 配置生成设置
config = GenerationConfig(
    batch_size=2,
    audio_format="flac",
)

result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")

# 访问结果
if result.success:
    for audio in result.audios:
        print(f"已生成：{audio['path']}")
        print(f"Key：{audio['key']}")
        print(f"Seed：{audio['params']['seed']}")
else:
    print(f"错误：{result.error}")

In [ ]:
#3.2.1主要函数

# 生成音乐的主函数 
def generate_music(
    dit_handler,
    llm_handler,
    params: GenerationParams,
    config: GenerationConfig,
    save_dir: Optional[str] = None,
    progress=None,
) -> GenerationResult

# 分析音频语义代码并提取元数据（captions、lyrics、BPM、调性等）。
def understand_music(
    llm_handler,
    audio_codes: str,
    temperature: float = 0.85,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.0,
    use_constrained_decoding: bool = True,
    constrained_decoding_debug: bool = False,
) -> UnderstandResult

# 从自然语言描述生成完整的音乐样本（caption、lyrics、元数据）。
def create_sample(
    llm_handler,
    query: str, #必需 | 期望音乐的自然语言描述
    instrumental: bool = False, #是否生成纯音乐
    vocal_language: Optional[str] = None, #将歌词限制为特定语言（例如"en"、"zh"、"bn"）
    temperature: float = 0.85, #采样温度
    top_k: Optional[int] = None, #Top-k 采样（None 禁用
    top_p: Optional[float] = None, #Top-p 采样（None 禁用
    repetition_penalty: float = 1.0, #重复惩罚
    use_constrained_decoding: bool = True, # 使用基于 FSM 的约束解码
    constrained_decoding_debug: bool = False,
) -> CreateSampleResult


# 格式化和增强用户提供的 caption 和 lyrics，生成结构化元数据。
def format_sample(
    llm_handler,
    caption: str,
    lyrics: str,
    user_metadata: Optional[Dict[str, Any]] = None,
    temperature: float = 0.85,
    top_k: Optional[int] = None,
    top_p: Optional[float] = None,
    repetition_penalty: float = 1.0,
    use_constrained_decoding: bool = True,
    constrained_decoding_debug: bool = False,
) -> FormatSampleResult

In [ ]:
#3.2.2配置对象--GenerationParams

#GenerationParams - 包含所有音乐生成参数：
@dataclass
class GenerationParams:
    # 任务和指令
    task_type: str = "text2music" #生成任务类型,详见3.5任务类型部分。
    instruction: str = "Fill the audio semantic mask based on the given conditions:" #任务特定。指令提示。
    
    # 音频上传
    reference_audio: Optional[str] = None #用于风格迁移或续写任务的参考音频文件路径。
    src_audio: Optional[str] = None #用于音频到音频任务（cover、repaint 等）的源音频文件路径。
    
    # LM 代码提示
    audio_codes: str = "" #预提取的 5Hz 音频语义代码字符串。仅供高级使用。
    
    # 文本输入
    caption: str = "" #期望音乐的文本描述。可以是简单提示如"放松的钢琴音乐"，或包含风格、情绪、乐器等的详细描述。最多 512 字符。
    lyrics: str = "" #人声音乐的歌词文本。纯音乐使用 `"[Instrumental]"`。支持多种语言【】。最多 4096 字符。
    instrumental: bool = False #如果为 True，无论歌词如何都生成纯音乐。
    
    # 元数据
    vocal_language: str = "unknown" #人声语言代码（ISO 639-1）。支持：`"en"`、`"zh"`、`"ja"`、`"es"`、`"fr"` 等。使用 `"unknown"` 自动检测。
    bpm: Optional[int] = None #每分钟节拍数（30-300）。`None` 启用通过 LM 自动检测。
    keyscale: str = "" #音乐调性（例如"C Major"、"Am"、"F# minor"）。空字符串启用自动检测。
    timesignature: str = "" #拍号（2 表示 '2/4'，3 表示 '3/4'，4 表示 '4/4'，6 表示 '6/8'）。空字符串启用自动检测。
    duration: float = -1.0 #目标音频长度（秒）（10-600）。如果 <= 0 或 None，模型根据歌词长度自动选择。
    
    # 高级设置
    inference_steps: int = 8 #去噪步数。Turbo 模型：1-20（推荐 8）。Base 模型：1-200（推荐 32-64）。越高 = 质量越好但更慢。|
    seed: int = -1 #用于可重复性的随机种子。使用 `-1` 表示随机种子，或任何正整数表示固定种子。
    guidance_scale: float = 7.0 #无分类器引导比例（1.0-15.0）。较高的值增加对文本提示的遵循度。仅支持非 turbo 模型。典型范围：5.0-9.0。
    use_adg: bool = False #使用自适应双引导（仅 base 模型）。以速度为代价提高质量。
    cfg_interval_start: float = 0.0 #CFG 应用起始比例（0.0-1.0）。控制何时开始应用无分类器引导。
    cfg_interval_end: float = 1.0 #CFG 应用结束比例（0.0-1.0）。控制何时停止应用无分类器引导。
    shift: float = 1.0 # 新增：时间步偏移因子 （范围 1.0-5.0，默认 1.0）。当 != 1.0 时，对时间步应用 `t = shift * t / (1 + (shift - 1) * t)`。turbo 模型推荐 3.0。
    infer_method: str = "ode"  # 新增：扩散推理方法 "ode"（Euler）更快且确定性。"sde"（随机）可能产生不同的带方差结果。
    timesteps: Optional[List[float]] = None  # 新增：自定义时间步，从 1.0 到 0.0 的浮点数列表（例如 `[0.97, 0.76, 0.615, 0.5, 0.395, 0.28, 0.18, 0.085, 0]`）。如果提供，覆盖 `inference_steps` 和 `shift`。
    
    repainting_start: float = 0.0 #重绘开始时间（秒）（用于 repaint/lego 任务）。
    repainting_end: float = -1 #重绘结束时间（秒）。使用 `-1` 表示音频末尾。
    audio_cover_strength: float = 1.0 #音频 cover/代码影响强度（0.0-1.0）。风格迁移任务设置较小值（0.2）。
    
    # 5Hz 语言模型参数
    thinking: bool = True #启用 5Hz 语言模型"思维链"推理用于语义/音乐元数据和代码。
    lm_temperature: float = 0.85 # LM 采样温度（0.0-2.0）。越高 = 更有创意/多样，越低 = 更保守。
    lm_cfg_scale: float = 2.0 #LM 无分类器引导比例。越高 = 更强的提示遵循度。
    lm_top_k: int = 0 #LM top-k 采样。`0` 禁用 top-k 过滤。典型值：40-100。
    lm_top_p: float = 0.9 #LM 核采样（0.0-1.0）。`1.0` 禁用核采样。典型值：0.9-0.95。
    lm_negative_prompt: str = "NO USER INPUT" #LM 引导的负面提示。帮助避免不想要的特征。
    use_cot_metas: bool = True #使用 LM CoT 推理生成元数据（BPM、调性、时长等）。
    use_cot_caption: bool = True #使用 LM CoT 推理优化用户 caption。
    use_cot_lyrics: bool = False #使用 LM CoT 推理检测人声语言。
    use_cot_language: bool = True #（保留供将来使用）使用 LM CoT 生成/优化歌词。
    use_constrained_decoding: bool = True #启用结构化 LM 输出的约束解码。
    
    # CoT 生成的值（在启用 CoT 推理时，由 LM 自动填充）
    cot_bpm: Optional[int] = None #LM 生成的 BPM 值。
    cot_keyscale: str = "" #LM 生成的调性。
    cot_timesignature: str = "" #LM 生成的拍号。
    cot_duration: Optional[float] = None #LM 生成的时长。
    cot_vocal_language: str = "unknown" #LM 检测的人声语言。
    cot_caption: str = "" #LM 优化的 caption。
    cot_lyrics: str = "" #LM 生成/优化的歌词。


In [ ]:
#3.2.2配置对象--GenerationConfig
#GenerationConfig:包含批处理和输出设置

@dataclass
class GenerationConfig:
    batch_size: int = 2 #并行生成的样本数量（1-8）。较高的值需要更多 GPU 内存。
    allow_lm_batch: bool = False #允许 LM 批处理。当 `batch_size >= 2` 且 `thinking=True` 时更快。
    use_random_seed: bool = True #是否使用随机种子。`True` 每次不同结果，`False` 可重复结果。
    seeds: Optional[List[int]] = None #批量生成的种子列表。如果提供的种子少于 batch_size，将用随机种子填充。也可以是单个 int。
    lm_batch_chunk_size: int = 8 # 每个 LM 推理块的最大批处理大小（GPU 内存限制）。
    constrained_decoding_debug: bool = False # 启用约束解码的调试日志。
    audio_format: str = "flac" #输出音频格式。选项：`"mp3"`、`"wav"`、`"flac"`。默认 FLAC 以快速保存。


In [ ]:
#3.2.3结果对象

In [ ]:
#3.5 任务类型

### 1. Text2Music（默认）：从文本描述和可选元数据生成音乐。

#关键参数**：

params = GenerationParams(
    task_type="text2music",
    caption="充满活力的摇滚音乐，电吉他",
    lyrics="[Instrumental]",  # 或实际歌词
    bpm=140,
    duration=30,
)
"""

**必需**：
- `caption` 或 `lyrics`（至少一个）

**可选但推荐**：
- `bpm`：控制节奏
- `keyscale`：控制音乐调性
- `timesignature`：控制节拍结构
- `duration`：控制长度
- `vocal_language`：控制人声特征

**用例**：
- 从文本描述生成音乐
- 从提示创建伴奏
- 生成带歌词的歌曲

"""

### 2. Cover：转换现有音频，保持作曲、和弦、结构但改变风格/音色。

params = GenerationParams(
    task_type="cover",
    src_audio="original_song.mp3",
    caption="爵士钢琴版本",
    audio_cover_strength=0.8,  # 0.0-1.0
)

"""
**必需**：
- `src_audio`：源音频文件路径
- `caption`：期望风格/转换的描述

**可选**：
- `audio_cover_strength`：控制原始音频的影响
  - `1.0`：强烈保持原始结构
  - `0.5`：平衡转换
  - `0.1`：宽松解读
- `lyrics`：新歌词（如果要更改人声）

**用例**：
- 创建不同风格的翻唱
- 在保持旋律的同时更改乐器
- 风格转换
"""

### 3. Repaint：重新生成音频的特定时间段，保持其余部分不变。


params = GenerationParams(
    task_type="repaint",
    src_audio="original.mp3",
    repainting_start=10.0,  # 秒
    repainting_end=20.0,    # 秒
    caption="带钢琴独奏的平滑过渡",
)


"""
**必需**：
- `src_audio`：源音频文件路径
- `repainting_start`：开始时间（秒）
- `repainting_end`：结束时间（秒）（使用 `-1` 表示文件末尾）
- `caption`：重绘部分期望内容的描述

**用例**：
- 修复生成音乐的特定部分
- 为歌曲的某些部分添加变化
- 创建平滑过渡
- 替换有问题的片段
"""

### 4. Lego（仅 Base 模型）：在现有音频的上下文中生成特定乐器轨道。


params = GenerationParams(
    task_type="lego",
    src_audio="backing_track.mp3",
    instruction="Generate the guitar track based on the audio context:",
    caption="带有蓝调感觉的主音吉他旋律",
    repainting_start=0.0,
    repainting_end=-1,
)

"""
**必需**：
- `src_audio`：源/伴奏音频路径
- `instruction`：必须指定轨道类型（例如"Generate the {TRACK_NAME} track..."）
- `caption`：期望轨道特征的描述

**可用轨道**：
- `"vocals"`、`"backing_vocals"`、`"drums"`、`"bass"`、`"guitar"`、`"keyboard"`、
- `"percussion"`、`"strings"`、`"synth"`、`"fx"`、`"brass"`、`"woodwinds"`

**用例**：
- 添加特定乐器轨道
- 在伴奏轨道上叠加额外乐器
- 迭代创建多轨作品

"""

### 5. Extract（仅 Base 模型）：从混音音频中提取/分离特定乐器轨道。

params = GenerationParams(
    task_type="extract",
    src_audio="full_mix.mp3",
    instruction="Extract the vocals track from the audio:",
)

"""
**必需**：
- `src_audio`：混音音频文件路径
- `instruction`：必须指定要提取的轨道

**可用轨道**：与 Lego 任务相同
- `"vocals"`、`"backing_vocals"`、`"drums"`、`"bass"`、`"guitar"`、`"keyboard"`、
- `"percussion"`、`"strings"`、`"synth"`、`"fx"`、`"brass"`、`"woodwinds"`

**用例**：
- 音轨分离
- 分离特定乐器
- 创建混音
- 分析单独轨道
"""

### 6. Complete（仅 Base 模型）：用指定的乐器完成/扩展部分轨道。

params = GenerationParams(
    task_type="complete",
    src_audio="incomplete_track.mp3",
    instruction="Complete the input track with drums, bass, guitar:",
    caption="摇滚风格完成",
)

"""
**必需**：
- `src_audio`：不完整/部分轨道的路径
- `instruction`：必须指定要添加的轨道
- `caption`：期望风格的描述

**用例**：
- 编排不完整的作品
- 添加伴奏轨道
- 自动完成音乐想法

"""

##3.6辅助函数

### understand_music

分析音频代码以提取音乐元数据。

```python
from acestep.inference import understand_music

result = understand_music(
    llm_handler=llm_handler,
    audio_codes="<|audio_code_123|><|audio_code_456|>...",
    temperature=0.85,
    use_constrained_decoding=True,
)

if result.success:
    print(f"Caption：{result.caption}")
    print(f"歌词：{result.lyrics}")
    print(f"BPM：{result.bpm}")
    print(f"调性：{result.keyscale}")
    print(f"时长：{result.duration}s")
    print(f"语言：{result.language}")
else:
    print(f"错误：{result.error}")
```

**用例**：
- 分析现有音乐
- 从音频代码提取元数据
- 逆向工程生成参数

---

### create_sample

从自然语言描述生成完整的音乐样本。这是"简单模式"/"灵感模式"功能。

```python
from acestep.inference import create_sample

result = create_sample(
    llm_handler=llm_handler,
    query="一首适合安静夜晚的柔和孟加拉情歌",
    instrumental=False,
    vocal_language="bn",  # 可选：限制为孟加拉语
    temperature=0.85,
)

if result.success:
    print(f"Caption：{result.caption}")
    print(f"歌词：{result.lyrics}")
    print(f"BPM：{result.bpm}")
    print(f"时长：{result.duration}s")
    print(f"调性：{result.keyscale}")
    print(f"是否纯音乐：{result.instrumental}")
    
    # 与 generate_music 一起使用
    params = GenerationParams(
        caption=result.caption,
        lyrics=result.lyrics,
        bpm=result.bpm,
        duration=result.duration,
        keyscale=result.keyscale,
        vocal_language=result.language,
    )
else:
    print(f"错误：{result.error}")
```

**参数**：

| 参数 | 类型 | 默认值 | 说明 |
|-----------|------|---------|-------------|
| `query` | `str` | 必需 | 期望音乐的自然语言描述 |
| `instrumental` | `bool` | `False` | 是否生成纯音乐 |
| `vocal_language` | `Optional[str]` | `None` | 将歌词限制为特定语言（例如"en"、"zh"、"bn"）|
| `temperature` | `float` | `0.85` | 采样温度 |
| `top_k` | `Optional[int]` | `None` | Top-k 采样（None 禁用）|
| `top_p` | `Optional[float]` | `None` | Top-p 采样（None 禁用）|
| `repetition_penalty` | `float` | `1.0` | 重复惩罚 |
| `use_constrained_decoding` | `bool` | `True` | 使用基于 FSM 的约束解码 |

---

### format_sample

格式化和增强用户提供的 caption 和 lyrics，生成结构化元数据。

```python
from acestep.inference import format_sample

result = format_sample(
    llm_handler=llm_handler,
    caption="拉丁流行，雷鬼音",
    lyrics="[Verse 1]\nBailando en la noche...",
    user_metadata={"bpm": 95},  # 可选：约束特定值
    temperature=0.85,
)

if result.success:
    print(f"增强后的 Caption：{result.caption}")
    print(f"格式化后的歌词：{result.lyrics}")
    print(f"BPM：{result.bpm}")
    print(f"时长：{result.duration}s")
    print(f"调性：{result.keyscale}")
    print(f"检测到的语言：{result.language}")
else:
    print(f"错误：{result.error}")
```

**参数**：

| 参数 | 类型 | 默认值 | 说明 |
|-----------|------|---------|-------------|
| `caption` | `str` | 必需 | 用户的 caption/描述 |
| `lyrics` | `str` | 必需 | 用户的带结构标签的歌词 |
| `user_metadata` | `Optional[Dict]` | `None` | 约束特定元数据值（bpm、duration、keyscale、timesignature、language）|
| `temperature` | `float` | `0.85` | 采样温度 |
| `top_k` | `Optional[int]` | `None` | Top-k 采样（None 禁用）|
| `top_p` | `Optional[float]` | `None` | Top-p 采样（None 禁用）|
| `repetition_penalty` | `float` | `1.0` | 重复惩罚 |
| `use_constrained_decoding` | `bool` | `True` | 使用基于 FSM 的约束解码 |



In [ ]:
## 3.7完整示例

### 示例 1：简单文本到音乐生成、器乐声

from acestep.inference import GenerationParams, GenerationConfig, generate_music

params = GenerationParams(
    task_type="text2music",
    caption="宁静的氛围音乐，柔和的钢琴和弦乐",
    duration=60,
    bpm=80,
    keyscale="C Major",

    #高级设置，turbo
    inference_steps = 8,
    seed =34,#随机种子数，-1代表随机。
    shift=3.0,  # Turbo 模型推荐
    infer_method = "ode",
    timesteps=[0.97, 0.76, 0.615, 0.5, 0.395, 0.28, 0.18, 0.085, 0],# 自定义 9 步调度
)

config = GenerationConfig(
    batch_size=2,  # 生成 2 个变体
    audio_format="flac",
)

result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")

if result.success:
    for i, audio in enumerate(result.audios, 1):
        print(f"变体 {i}：{audio['path']}")


In [17]:
### 示例 5：使用 create_sample 的简单模式

from acestep.inference import create_sample, GenerationParams, GenerationConfig, generate_music

# 步骤 1：从描述创建样本
sample = create_sample(
    llm_handler=llm_handler,
    query="欢快的摇滚民谣",
    vocal_language="zh",#人声语言代码（ISO 639-1）。支持：`"en"`、`"zh"`、`"ja"`、`"es"`、`"fr"` 等。使用 `"unknown"` 自动检测。
)

if sample.success:

    
    print(f"Caption：{sample.caption}")
    print(f"歌词：{sample.lyrics}")
    print(f"BPM：{sample.bpm}")
    print(f"时长：{sample.duration}s")
    print(f"调性：{sample.keyscale}")
    print(f"是否纯音乐：{sample.instrumental}")
    # 步骤 2：使用样本生成音乐
    params = GenerationParams(
        caption=sample.caption,
        lyrics=sample.lyrics,
        bpm=sample.bpm,
        duration=sample.duration,
        keyscale=sample.keyscale,
        vocal_language=sample.language,
        thinking=True,
    )
    
    config = GenerationConfig(batch_size=1,audio_format="mp3",)
    result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")


2026-02-07 17:07:24.290 | INFO     | acestep.llm_inference:create_sample_from_query:1630 - Creating sample from query: 欢快的摇滚民谣... (instrumental=False, vocal_language=zh)
2026-02-07 17:07:24.300 | DEBUG    | acestep.llm_inference:create_sample_from_query:1637 - Formatted prompt for inspiration: <|im_start|>system
# Instruction
Expand the user's input into a more detailed and specific musical description:

<|im_end|>
<|im_start|>user
欢快的摇滚民谣

instrumental: false<|im_end|>
<|im_start|>assistant

2026-02-07 17:07:24.301 | INFO     | acestep.llm_inference:create_sample_from_query:1646 - Using user-specified language: zh
LLM Constrained Decoding:  16%|█▌        | 645/4032 [00:27<02:25, 23.22token/s]
2026-02-07 17:07:52.092 | DEBUG    | acestep.llm_inference:parse_lm_output:2261 - Debug output text: <think>
bpm: 143
caption: An upbeat and energetic pop-rock track driven by a bright acoustic guitar
  strumming a catchy chord progression. A clean, melodic electric guitar plays lead
  lines and 

Caption：An upbeat and energetic pop-rock track driven by a bright acoustic guitar strumming a catchy chord progression. A clean, melodic electric guitar plays lead lines and fills, complementing the enthusiastic male lead vocal. The rhythm section consists of a solid, driving bassline and a crisp, live-sounding drum kit providing a steady rock beat. The production is polished and radio-friendly, with a celebratory and joyful mood throughout, culminating in a short, melodic guitar break before a final, powerful chorus.
歌词：[Intro]
(Oh-oh-oh-oh-oh-oh-oh-oh)
(Oh-oh-oh-oh-oh-oh-oh-oh)

[Verse 1]
期待，这圣诞时候
让欢乐充满旧梦
旧梦，就是时候
新的快乐不会走
人生，足有，心变
你我，我们来将它舞动
今夜为你，唱响
欢乐伴着你，舞动

[Verse 2]
今晚，大家来相伴
共享欢乐不孤单
谁的，孤单不孤单
大家都有şar işar
今晚，大家来相伴
共享欢乐不孤单
每个，孤独的灵魂
都有şar işar

[Chorus]
今夜为你，我的ingles
为你带来无数欢乐
希望你的每个瞬间
都能带来无限笑容

[Post-Chorus]
(Oh-oh-oh-oh-oh-oh-oh-oh)
(Oh-oh-oh-oh-oh-oh-oh-oh)

[Verse 3]
青春年少，让烦恼
通通飘散在风里
美好时光，莫错过
与爱共舞永不寂寞
人生，足有，心变
你我，我们来将它舞动
今夜为你，唱响
欢乐伴着你，舞动

[Verse 4]
今晚，大家来相伴
共享欢乐不孤单
谁的，孤单不孤单
大家都有şar

LLM CFG Generation:   4%|▎         | 145/4032 [00:08<03:51, 16.80token/s]
2026-02-07 17:08:00.748 | DEBUG    | acestep.llm_inference:parse_lm_output:2261 - Debug output text: <think>
bpm: 143
caption: An upbeat and energetic pop-rock track driven by a bright acoustic guitar
  strumming a catchy chord progression. A clean, melodic electric guitar plays lead
  lines and fills, complementing the enthusiastic male lead vocal. The rhythm section
  consists of a solid, driving bassline and a crisp, live-sounding drum kitprecio.
  The production is polished and radio-friendly, with a celebratory and joyful mood
  throughout, culminating in a short, melodic guitar break before a final, powerful
  chorus.
duration: 223
keyscale: A major
language: zh
timesignature: 4
<|im_end|>
2026-02-07 17:08:00.749 | INFO     | acestep.llm_inference:generate_with_stop_condition:1016 - Phase 1 completed in 8.65s. Generated metadata: ['bpm', 'caption', 'duration', 'keyscale', 'language', 'timesignature']
2026-0

Using precomputed LM hints
Using precomputed LM hints


2026-02-07 17:09:00.861 | INFO     | acestep.handler:generate_music:2916 - [generate_music] Model generation completed. Decoding latents...
2026-02-07 17:09:00.948 | DEBUG    | acestep.handler:generate_music:2920 - [generate_music] pred_latents: torch.Size([1, 5575, 64]), dtype=torch.bfloat16 pred_latents.min()=tensor(-6.8125, device='cuda:0', dtype=torch.bfloat16), pred_latents.max()=tensor(4.3750, device='cuda:0', dtype=torch.bfloat16), pred_latents.mean()=tensor(-0.0201, device='cuda:0', dtype=torch.bfloat16) pred_latents.std()=tensor(0.9922, device='cuda:0', dtype=torch.bfloat16)
2026-02-07 17:09:00.949 | DEBUG    | acestep.handler:generate_music:2921 - [generate_music] time_costs: {'encoder_time_cost': 0.018764019012451172, 'diffusion_time_cost': 1.2205278873443604, 'diffusion_per_step_time_cost': 0.15256598591804504, 'total_time_cost': 1.2392919063568115, 'offload_time_cost': 0.0}
2026-02-07 17:09:00.950 | INFO     | acestep.handler:generate_music:2924 - [generate_music] Decoding

In [ ]:
### 示例 2：带歌词的歌曲生成

#-------------------漏字了--------------------------
params = GenerationParams(
    task_type="text2music",
    caption="流行民谣，情感人声",
    lyrics="""Verse 1:
今天走在街上
想着你曾说过的话
一切都变得不同了
但我会找到自己的路

Chorus:
我在前进，我很坚强
这就是我属于的地方
""",
    vocal_language="zh",
    bpm=72,
    duration=45,
)

config = GenerationConfig(batch_size=1)

result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")

2026-02-07 15:59:40.700 | INFO     | acestep.inference:generate_music:387 - [generate_music] LLM usage decision: thinking=True, use_cot_caption=True, use_cot_language=True, use_cot_metas=True, need_lm_for_cot=True, llm_initialized=True, use_lm=True
2026-02-07 15:59:40.703 | INFO     | acestep.inference:generate_music:445 - LM chunk 1/1 (infer_type=llm_dit) (size: 1, seeds: [4024853432])
2026-02-07 15:59:40.704 | INFO     | acestep.llm_inference:generate_with_stop_condition:968 - Phase 1: Generating CoT metadata...
2026-02-07 15:59:40.710 | INFO     | acestep.llm_inference:generate_with_stop_condition:974 - generate_with_stop_condition: formatted_prompt=<|im_start|>system
# Instruction
Generate audio semantic tokens based on the given conditions:

<|im_end|>
<|im_start|>user
# Caption
流行民谣，情感人声

# Lyric
Verse 1:
今天走在街上
想着你曾说过的话
一切都变得不同了
但我会找到自己的路

Chorus:
我在前进，我很坚强
这就是我属于的地方

<|im_end|>
<|im_start|>assistant

LLM CFG Generation:   4%|▍         | 165/4032 [00:08<03:20, 19.27token/s]
2026

Using precomputed LM hints
Using precomputed LM hints


2026-02-07 16:00:02.031 | INFO     | acestep.handler:generate_music:2916 - [generate_music] Model generation completed. Decoding latents...
2026-02-07 16:00:02.063 | DEBUG    | acestep.handler:generate_music:2920 - [generate_music] pred_latents: torch.Size([1, 1125, 64]), dtype=torch.bfloat16 pred_latents.min()=tensor(-7.0312, device='cuda:0', dtype=torch.bfloat16), pred_latents.max()=tensor(4.2188, device='cuda:0', dtype=torch.bfloat16), pred_latents.mean()=tensor(-0.1123, device='cuda:0', dtype=torch.bfloat16) pred_latents.std()=tensor(1.0234, device='cuda:0', dtype=torch.bfloat16)
2026-02-07 16:00:02.064 | DEBUG    | acestep.handler:generate_music:2921 - [generate_music] time_costs: {'encoder_time_cost': 0.018654823303222656, 'diffusion_time_cost': 0.4355146884918213, 'diffusion_per_step_time_cost': 0.05443933606147766, 'total_time_cost': 0.45416951179504395, 'offload_time_cost': 0.0}
2026-02-07 16:00:02.064 | INFO     | acestep.handler:generate_music:2924 - [generate_music] Decodin

In [ ]:
### 示例 6：格式化和增强用户输入

from acestep.inference import format_sample, GenerationParams, GenerationConfig, generate_music

# 步骤 1：格式化用户输入
formatted = format_sample(
    llm_handler=llm_handler,
    caption="摇滚民谣",
    lyrics="[Verse]\n在黑暗中我找到了自己的路...",
)

if formatted.success:
    # 步骤 2：使用增强后的输入生成
    params = GenerationParams(
        caption=formatted.caption,
        lyrics=formatted.lyrics,
        bpm=formatted.bpm,
        duration=formatted.duration,
        keyscale=formatted.keyscale,
        thinking=True,
        use_cot_metas=False,  # 已格式化，跳过元数据 CoT
    )
    
    config = GenerationConfig(batch_size=2)
    result = generate_music(dit_handler, llm_handler, params, config, save_dir="/output")

In [44]:
### Cover：转换现有音频，保持作曲、和弦、结构但改变风格/音色。
from acestep.inference import create_sample, GenerationParams, GenerationConfig, generate_music

params = GenerationParams(
    task_type="cover",
    src_audio="/srv/home/guxin/ACE-Step-1.5-main/output/0ef5ee92-83cf-7cce-86b8-8476dd4f5a9a.mp3",
    caption= "",#"悲伤、忧郁版本",#"爵士钢琴版本"
    lyrics="""
    Verse 1:
    今天走在街上
    想着你曾说过的话
    一切都变得不同了
    但我会找到自己的路

    Chorus:
    我在前进，我很坚强
    这就是我属于的地方

    Verse 1:
    今天走在街上
    想着我们曾聊过的天
    一切都好像变得相同了
    我希望找到我们的路
    """,
    audio_cover_strength=0.5,# 0.0-1.0越小则改动越大
    
)

"""
    lyrics=sample.lyrics,
    bpm=sample.bpm,
    duration=sample.duration,
    keyscale=sample.keyscale,
    vocal_language=sample.language,
"""
config = GenerationConfig(batch_size=1,audio_format="mp3",)
result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")

"""
**必需**：
- `src_audio`：源音频文件路径
- `caption`：期望风格/转换的描述

**可选**：
- `audio_cover_strength`：控制原始音频的影响
  - `1.0`：强烈保持原始结构
  - `0.5`：平衡转换
  - `0.1`：宽松解读
- `lyrics`：新歌词（如果要更改人声）

**用例**：
- 创建不同风格的翻唱
- 在保持旋律的同时更改乐器
- 风格转换
"""

2026-02-07 19:59:04.878 | INFO     | acestep.inference:generate_music:385 - Skipping LM for task_type='cover' - using DiT directly
2026-02-07 19:59:04.879 | INFO     | acestep.inference:generate_music:387 - [generate_music] LLM usage decision: thinking=True, use_cot_caption=True, use_cot_language=True, use_cot_metas=True, need_lm_for_cot=True, llm_initialized=True, use_lm=False
2026-02-07 19:59:04.880 | INFO     | acestep.handler:generate_music:2805 - [generate_music] Starting generation...
2026-02-07 19:59:04.881 | INFO     | acestep.handler:generate_music:2808 - [generate_music] Preparing inputs...
2026-02-07 19:59:04.882 | INFO     | acestep.handler:generate_music:2849 - [generate_music] Processing source audio...
2026-02-07 19:59:05.390 | INFO     | acestep.handler:_prepare_batch:1683 - [generate_music] Encoding target audio to latents for item 0...
Encoding audio chunks: 100%|██████████| 8/8 [00:01<00:00,  5.34it/s]
2026-02-07 19:59:07.108 | INFO     | acestep.handler:_prepare_bat

'\n**必需**：\n- `src_audio`：源音频文件路径\n- `caption`：期望风格/转换的描述\n\n**可选**：\n- `audio_cover_strength`：控制原始音频的影响\n  - `1.0`：强烈保持原始结构\n  - `0.5`：平衡转换\n  - `0.1`：宽松解读\n- `lyrics`：新歌词（如果要更改人声）\n\n**用例**：\n- 创建不同风格的翻唱\n- 在保持旋律的同时更改乐器\n- 风格转换\n'

In [ ]:
#cover-----英文版本提供了更多信息

params = GenerationParams(
    task_type="cover",
    src_audio="original_pop_song.mp3",
    caption="orchestral symphonic arrangement",
    audio_cover_strength=0.7,
    thinking=True,  # Enable LM for metadata
    use_cot_metas=True,
)

config = GenerationConfig(batch_size=1)

result = generate_music(dit_handler, llm_handler, params, config, save_dir="/output")

# Access LM-generated metadata
if result.extra_outputs.get("lm_metadata"):
    lm_meta = result.extra_outputs["lm_metadata"]
    print(f"LM detected BPM: {lm_meta.get('bpm')}")
    print(f"LM detected Key: {lm_meta.get('keyscale')}")

In [ ]:
### 3. Repaint：重新生成音频的特定时间段，保持其余部分不变。

params = GenerationParams(
    task_type="repaint",
    src_audio="/srv/home/guxin/ACE-Step-1.5-main/output/0ef5ee92-83cf-7cce-86b8-8476dd4f5a9a.mp3",
    repainting_start=48.0,  # 秒
    repainting_end=78.0,    # 秒
    caption="带钢琴独奏的平滑过渡",
)

config = GenerationConfig(batch_size=1,audio_format="mp3",)
result = generate_music(dit_handler, llm_handler, params, config, save_dir="/srv/home/guxin/ACE-Step-1.5-main/output")


"""
**必需**：
- `src_audio`：源音频文件路径
- `repainting_start`：开始时间（秒）
- `repainting_end`：结束时间（秒）（使用 `-1` 表示文件末尾）
- `caption`：重绘部分期望内容的描述

**用例**：
- 修复生成音乐的特定部分
- 为歌曲的某些部分添加变化
- 创建平滑过渡
- 替换有问题的片段
"""

2026-02-07 20:10:15.573 | INFO     | acestep.inference:generate_music:385 - Skipping LM for task_type='repaint' - using DiT directly
2026-02-07 20:10:15.575 | INFO     | acestep.inference:generate_music:387 - [generate_music] LLM usage decision: thinking=True, use_cot_caption=True, use_cot_language=True, use_cot_metas=True, need_lm_for_cot=True, llm_initialized=True, use_lm=False
2026-02-07 20:10:15.576 | INFO     | acestep.handler:generate_music:2805 - [generate_music] Starting generation...
2026-02-07 20:10:15.577 | INFO     | acestep.handler:generate_music:2808 - [generate_music] Preparing inputs...
2026-02-07 20:10:15.578 | INFO     | acestep.handler:generate_music:2849 - [generate_music] Processing source audio...
2026-02-07 20:10:16.091 | INFO     | acestep.handler:_prepare_batch:1683 - [generate_music] Encoding target audio to latents for item 0...
Encoding audio chunks: 100%|██████████| 8/8 [00:01<00:00,  5.34it/s]
2026-02-07 20:10:17.810 | INFO     | acestep.handler:_prepare_b

'\n**必需**：\n- `src_audio`：源音频文件路径\n- `repainting_start`：开始时间（秒）\n- `repainting_end`：结束时间（秒）（使用 `-1` 表示文件末尾）\n- `caption`：重绘部分期望内容的描述\n\n**用例**：\n- 修复生成音乐的特定部分\n- 为歌曲的某些部分添加变化\n- 创建平滑过渡\n- 替换有问题的片段\n'